In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tensorflow opencv-python matplotlib pandas scikit-learn

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
!mkdir /content/skin_dataset

In [ ]:
!unzip -q "/content/drive/MyDrive/Skin_lesion/raw_data/HAM10000_images_part_1.zip" -d /content/skin_dataset
!unzip -q "/content/drive/MyDrive/Skin_lesion/raw_data/HAM10000_images_part_2.zip" -d /content/skin_dataset

In [ ]:
import pandas as pd

tab_path = "/content/drive/MyDrive/Skin_lesion/raw_data/HAM10000_metadata.tab"
df = pd.read_csv(tab_path, sep='\t')

csv_path = "/content/skin_dataset/HAM10000_metadata.csv"
df.to_csv(csv_path, index=False)

print("Converted to CSV successfully")

Converted to CSV successfully


In [ ]:
import os

print("Number of images:",
      len([f for f in os.listdir('/content/skin_dataset') if f.endswith('.jpg')]))

Number of images: 10015


In [ ]:
IMAGE_DIR = "/content/skin_dataset"
CSV_PATH = "/content/skin_dataset/HAM10000_metadata.csv"

In [ ]:
import os
imgs = [f for f in os.listdir("/content/skin_dataset") if f.lower().endswith(".jpg")]
print("Images found:", len(imgs))
print("Sample:", imgs[:5])

Images found: 10015
Sample: ['ISIC_0034044.jpg', 'ISIC_0031117.jpg', 'ISIC_0025271.jpg', 'ISIC_0031568.jpg', 'ISIC_0031914.jpg']


In [ ]:
import pandas as pd

TAB_PATH = "/content/drive/MyDrive/Skin_lesion/raw_data/HAM10000_metadata.tab"
df = pd.read_csv(TAB_PATH, sep="\t")
print(df.shape)
df.head()

(10015, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["dx"])

print("Classes:", list(le.classes_))
df[["image_id", "dx", "label"]].head()

Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


,image_id,dx,label
0,ISIC_0027419,bkl,2
1,ISIC_0025030,bkl,2
2,ISIC_0026769,bkl,2
3,ISIC_0025661,bkl,2
4,ISIC_0031633,bkl,2


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

print("Train:", len(train_df), "Test:", len(test_df))

Train: 8012 Test: 2003


In [ ]:
import os

train_df = train_df.copy()
test_df = test_df.copy()

train_df["path"] = train_df["image_id"].apply(lambda x: os.path.join(IMAGE_DIR, x + ".jpg"))
test_df["path"]  = test_df["image_id"].apply(lambda x: os.path.join(IMAGE_DIR, x + ".jpg"))

In [ ]:
import os

IMAGE_DIR = "/content/skin_dataset"
train_df["path"] = train_df["image_id"].apply(lambda x: os.path.join(IMAGE_DIR, x + ".jpg"))
test_df["path"]  = test_df["image_id"].apply(lambda x: os.path.join(IMAGE_DIR, x + ".jpg"))

# keep only existing files (safety)
train_df = train_df[train_df["path"].apply(os.path.exists)]
test_df  = test_df[test_df["path"].apply(os.path.exists)]

print("Train usable:", len(train_df), "Test usable:", len(test_df))

Train usable: 8012 Test usable: 2003


In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def decode_resize_normalize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

In [ ]:
augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
])

In [ ]:
def make_dataset(df, training=True):
    paths = df["path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(2000)

    ds = ds.map(decode_resize_normalize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)

    if training:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)

    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
test_ds  = make_dataset(test_df, training=False)

print("Datasets ready ✅")

Datasets ready ✅


In [ ]:
base = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(7, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5),
]

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras", monitor="val_loss", save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", patience=4, factor=0.5, min_lr=1e-7
    ),
]

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=100,
    callbacks=callbacks
)

Epoch 1/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 660s 1s/step - accuracy: 0.6706 - loss: 1.0543 - val_accuracy: 0.7369 - val_loss: 0.7322 - learning_rate: 0.0010
Epoch 2/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 644s 1s/step - accuracy: 0.7247 - loss: 0.7754 - val_accuracy: 0.7184 - val_loss: 0.8052 - learning_rate: 0.0010
Epoch 3/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 647s 1s/step - accuracy: 0.7333 - loss: 0.7491 - val_accuracy: 0.7494 - val_loss: 0.7117 - learning_rate: 0.0010
Epoch 4/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 594s 1s/step - accuracy: 0.7374 - loss: 0.7233 - val_accuracy: 0.7494 - val_loss: 0.7310 - learning_rate: 0.0010
Epoch 5/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 646s 1s/step - accuracy: 0.7466 - loss: 0.6935 - val_accuracy: 0.7509 - val_loss: 0.7011 - learning_rate: 0.0010
Epoch 6/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 598s 1s/step - accuracy: 0.7526 - loss: 0.6757 - val_accuracy: 0.7494 - val_loss: 0.7392 - learning_rate: 0.0010
Epoch 7/100
501/501 ━━━━━━━━━━━━━━━━━━━━ 638s 1s/step - accuracy: 0.7462 - l

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=50,          # you can set 100 also
    callbacks=callbacks
)

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("best_model.keras")
print("Best model loaded ✅")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Accuracy curve
plt.plot(history2.history['accuracy'], label='Train Accuracy')
plt.plot(history2.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Accuracy vs Epoch")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.show()

# Loss curve
plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Loss vs Epoch")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

In [ ]:
model.save("skin_lesion_model.keras")
print("Model saved successfully ✅")